# Unit 4 Hands-On ②: REINFORCE — Pixelcopter 실습

이 노트북은 **Hugging Face 딥 강화학습 강좌 Unit 4**의 두 번째 실습입니다.  
CartPole-v1과 동일한 **REINFORCE** 알고리즘을 사용하지만,  
환경이 `Pixelcopter-PLE-v0`으로 훨씬 어렵습니다.

### CartPole과의 차이점

| 항목 | CartPole-v1 | Pixelcopter-PLE-v0 |
|---|---|---|
| **상태 공간** | 4차원 (연속값) | 7차원 (연속값) |
| **행동 공간** | 2개 (좌/우) | 2개 (위/아무것도 안 함) |
| **신경망** | 2층 MLP | **3층 MLP** (더 복잡한 환경 반영) |
| **훈련 에피소드** | 1,000 | **10,000** (더 어려운 환경) |
| **gamma** | 1.0 | **0.99** (장기 보상 할인) |
| **gym 버전** | gymnasium | gym (구버전, gym-pygame 사용) |

### Pixelcopter란?
```
       🚁  ← 헬리콥터
  ____      ____
       |  |
       |  |   ← 장애물 (천장/바닥)
  ____|  |____
```
헬리콥터를 조종하여 장애물을 피하고 최대한 오래 생존하는 환경입니다.

---
## 목차
1. 환경 설치
2. Google Drive 마운트
3. 가상 디스플레이 설정
4. 라이브러리 임포트
5. 환경 탐색
6. 정책 네트워크 정의
7. REINFORCE 훈련 함수
8. 하이퍼파라미터 설정 및 훈련
9. 훈련 곡선 시각화
10. 에이전트 평가
11. 훈련 과정 영상 저장
12. Hugging Face Hub 업로드


---
## 1. 환경 설치

Pixelcopter는 `gym-pygame` 패키지를 통해 제공됩니다.  

> ⚠️ **numpy 버전 주의**: `gym-pygame`은 numpy 1.26.4 이하가 필요합니다.  
> 설치 후 **런타임을 재시작**해야 변경된 numpy가 적용됩니다.


In [1]:
!apt install -y python-opengl ffmpeg xvfb -qq
!pip install pyvirtualdisplay imageio imageio-ffmpeg huggingface_hub -q

E: Unable to locate package python-opengl


In [ ]:
# numpy / gym / gym-pygame 호환성 재설치
import sys
import subprocess

# 기존 설치된 호환성 문제 패키지 제거
for pkg in ['numpy', 'gym', 'gym-pygame']:
    try:
        subprocess.check_call([sys.executable, '-m', 'pip', 'uninstall', '-y', pkg],
                              stdout=subprocess.DEVNULL,
                              stderr=subprocess.DEVNULL)
    except Exception:
        pass

# 안정적인 조합으로 재설치
!{sys.executable} -m pip install --upgrade --force-reinstall "numpy==1.26.4" "gym==0.26.2" "gym-pygame==0.0.2" "pygame==2.5.2" -q

print('✅ 패키지 재설치 완료')

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 721.7/721.7 kB 17.3 MB/s eta 0:00:0000:01
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
ERROR: Ignored the following versions that require a different python version: 1.21.2 Requires-Python >=3.7,<3.11; 1.21.3 Requires-Python >=3.7,<3.11; 1.21.4 Requires-Python >=3.7,<3.11; 1.21.5 Requires-Python >=3.7,<3.11; 1.21.6 Requires-Python >=3.7,<3.11
ERROR: Could not find a version that satisfies the requirement gym-pygame==0.0.2 (from versions: none)
ERROR: No matching distribution found for gym-pygame==0.0.2
✅ 패키지 재설치 완료
⚠️ 런타임을 재시작한 뒤, 위에서부터 다시 실행하세요.


---
## 2. Google Drive 마운트

Colab VM은 세션 종료 시 파일이 삭제됩니다.  
훈련 영상과 모델을 Google Drive에 저장합니다.

```
Google Drive/RL_Course/Unit4_Pixelcopter/
├── training_videos/   ← 훈련 완료 후 영상
└── model.pt           ← 최종 모델 백업
```


In [ ]:
from google.colab import drive
import os

drive.mount('/content/drive')

# ✏️ 저장 폴더명을 원하는 대로 변경하세요.
DRIVE_BASE = '/content/drive/MyDrive/RL_Course/Unit4_Pixelcopter'
VIDEO_DIR  = f'{DRIVE_BASE}/training_videos'
MODEL_DIR  = DRIVE_BASE

os.makedirs(VIDEO_DIR, exist_ok=True)
os.makedirs(MODEL_DIR, exist_ok=True)

print('✅ Drive 마운트 완료!')
print(f'   영상 저장 경로 : {VIDEO_DIR}')
print(f'   모델 저장 경로 : {MODEL_DIR}')


Mounted at /content/drive
✅ Drive 마운트 완료!
   영상 저장 경로 : /content/drive/MyDrive/RL_Course/Unit4_Pixelcopter/training_videos
   모델 저장 경로 : /content/drive/MyDrive/RL_Course/Unit4_Pixelcopter


---
## 3. 가상 디스플레이 설정


In [ ]:
from pyvirtualdisplay import Display

virtual_display = Display(visible=0, size=(1400, 900))
virtual_display.start()
print('✅ 가상 디스플레이 시작')


✅ 가상 디스플레이 시작


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


---
## 4. 라이브러리 임포트

| 라이브러리 | 역할 |
|---|---|
| `gym` + `gym_pygame` | Pixelcopter 환경 (구버전 gym API) |
| `torch` | 신경망 정의 및 역전파 |
| `imageio` | 프레임을 mp4로 저장 |

> ⚠️ Pixelcopter는 `gymnasium`이 아닌 구버전 **`gym`** 을 사용합니다.  
> API 차이: `env.reset()` → 상태만 반환 (info 없음)  
> `env.step()` → `(state, reward, done, info)` 4개 반환 (terminated/truncated 분리 없음)


In [ ]:
import numpy as np
from collections import deque

import matplotlib.pyplot as plt
%matplotlib inline

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.distributions import Categorical

# gym-pygame: Pixelcopter 환경 제공
import gym
import gym_pygame
import imageio

from huggingface_hub import HfApi, login
from huggingface_hub.repocard import metadata_eval_result, metadata_save
from IPython.display import Video, display
from pathlib import Path
import datetime, json, tempfile


ModuleNotFoundError: No module named 'gym'

: 

: 

In [ ]:
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
print('사용 디바이스:', device)


---
## 5. 환경 탐색

**관측 공간** (7차원 연속값):
- 헬리콥터 y 위치 / y 속도
- 다음 블록까지의 거리 / 블록 높이
- 다음 블록 상단 y / 다음 블록 하단 y
- 헬리콥터 현재 높이

**행동 공간** (2개):
- 0: 아무것도 하지 않음 (중력으로 하강)
- 1: 엔진 점화 (상승)

**보상**: 살아남은 매 프레임 +1, 충돌 시 에피소드 종료


In [ ]:
env_id = 'Pixelcopter-PLE-v0'
env = gym.make(env_id)
eval_env = gym.make(env_id)

s_size = env.observation_space.shape[0]  # 7
a_size = env.action_space.n              # 2

print('===== 관측 공간(Observation Space) =====')
print('크기:', s_size)
print('샘플:', env.observation_space.sample())

print('\n===== 행동 공간(Action Space) =====')
print('크기:', a_size, '→ 0: 하강(대기), 1: 상승(엔진점화)')


---
## 6. 정책 네트워크 (Policy Network)

Pixelcopter는 CartPole보다 복잡한 환경이므로 **3층 MLP**를 사용합니다.

```
입력 (상태 s, 7차원)
     │
  FC Layer 1  (7 → h_size,    ReLU)
     │
  FC Layer 2  (h_size → h_size×2, ReLU)  ← CartPole에 없는 추가 레이어
     │
  FC Layer 3  (h_size×2 → 2)
     │
  Softmax  → [P(하강), P(상승)]
```


In [ ]:
class Policy(nn.Module):
    def __init__(self, s_size, a_size, h_size):
        super(Policy, self).__init__()
        # CartPole(2층)보다 레이어를 하나 더 추가 (더 복잡한 환경 대응)
        self.fc1 = nn.Linear(s_size, h_size)
        self.fc2 = nn.Linear(h_size, h_size * 2)
        self.fc3 = nn.Linear(h_size * 2, a_size)

    def forward(self, x):
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = self.fc3(x)
        return F.softmax(x, dim=1)  # 행동 확률 반환

    def act(self, state):
        """상태를 받아 행동과 log_prob 반환"""
        state = torch.from_numpy(state).float().unsqueeze(0).to(device)
        probs = self.forward(state).cpu()
        m = Categorical(probs)
        action = m.sample()
        return action.item(), m.log_prob(action)


---
## 7. REINFORCE 훈련 함수

CartPole과 동일한 REINFORCE 알고리즘이지만,  
구버전 gym API에 맞게 `env.reset()` / `env.step()` 반환값 처리가 다릅니다.

| | CartPole (gymnasium) | Pixelcopter (gym) |
|---|---|---|
| `reset()` 반환 | `state, info` | `state` (info 없음) |
| `step()` 반환 | `state, reward, terminated, truncated, info` | `state, reward, done, info` |


In [ ]:
def reinforce(policy, optimizer, n_training_episodes, max_t, gamma, print_every):
    """
    REINFORCE 훈련 루프 (구버전 gym API 사용)

    Args:
        policy: 정책 네트워크
        optimizer: 옵티마이저
        n_training_episodes: 총 훈련 에피소드 수
        max_t: 에피소드당 최대 스텝
        gamma: 할인율
        print_every: 몇 에피소드마다 로그 출력
    """
    scores_deque = deque(maxlen=100)
    scores = []

    for i_episode in range(1, n_training_episodes + 1):
        saved_log_probs = []
        rewards = []

        # ⚠️ 구버전 gym: reset()은 state만 반환 (info 없음)
        state = env.reset()

        for t in range(max_t):
            action, log_prob = policy.act(state)
            saved_log_probs.append(log_prob)

            # ⚠️ 구버전 gym: step()은 (state, reward, done, info) 반환
            #    gymnasium처럼 terminated/truncated로 분리되지 않음
            state, reward, done, _ = env.step(action)
            rewards.append(reward)
            if done:
                break

        scores_deque.append(sum(rewards))
        scores.append(sum(rewards))

        # ── 누적 할인 보상(Return) 계산 (역방향 O(N)) ──────────────
        returns = deque(maxlen=max_t)
        for t in range(len(rewards))[::-1]:
            disc_return_t = returns[0] if len(returns) > 0 else 0
            returns.appendleft(gamma * disc_return_t + rewards[t])

        # ── Return 정규화 ──────────────────────────────────────────
        eps = np.finfo(np.float32).eps.item()
        returns = torch.tensor(returns)
        returns = (returns - returns.mean()) / (returns.std() + eps)

        # ── 정책 손실 계산 및 역전파 ───────────────────────────────
        policy_loss = torch.cat(
            [-log_prob * G for log_prob, G in zip(saved_log_probs, returns)]
        ).sum()

        optimizer.zero_grad()
        policy_loss.backward()
        optimizer.step()

        if i_episode % print_every == 0:
            print(f'에피소드 {i_episode}\t최근 100회 평균 점수: {np.mean(scores_deque):.2f}')

    return scores


---
## 8. 하이퍼파라미터 설정 및 훈련

| 파라미터 | 값 | CartPole 대비 |
|---|---|---|
| `h_size` | 64 | CartPole(16)보다 4배 큰 은닉층 |
| `n_training_episodes` | 10,000 | CartPole(1,000)의 10배 |
| `max_t` | 10,000 | CartPole(1,000)의 10배 |
| `gamma` | 0.99 | CartPole(1.0)보다 미래 보상 소폭 할인 |
| `lr` | 1e-4 | CartPole(1e-2)보다 작은 학습률 (안정적 수렴) |

> ⏱️ 예상 훈련 시간: Colab GPU 기준 **약 30~60분**


In [ ]:
pixelcopter_hyperparameters = {
    'h_size': 64,
    'n_training_episodes': 10_000,
    'n_evaluation_episodes': 10,
    'max_t': 10_000,
    'gamma': 0.99,
    'lr': 1e-4,
    'env_id': env_id,
    'state_space': s_size,
    'action_space': a_size,
}


In [ ]:
# 정책 네트워크 및 옵티마이저 생성
pixelcopter_policy = Policy(
    pixelcopter_hyperparameters['state_space'],
    pixelcopter_hyperparameters['action_space'],
    pixelcopter_hyperparameters['h_size'],
).to(device)

pixelcopter_optimizer = optim.Adam(
    pixelcopter_policy.parameters(),
    lr=pixelcopter_hyperparameters['lr']
)

# 훈련 실행
scores = reinforce(
    pixelcopter_policy,
    pixelcopter_optimizer,
    pixelcopter_hyperparameters['n_training_episodes'],
    pixelcopter_hyperparameters['max_t'],
    pixelcopter_hyperparameters['gamma'],
    print_every=1000,
)


---
## 9. 훈련 곡선 시각화


In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

ax.plot(scores, alpha=0.3, label='에피소드 점수')

if len(scores) >= 100:
    moving_avg = [np.mean(scores[max(0, i-99):i+1]) for i in range(len(scores))]
    ax.plot(moving_avg, label='100 에피소드 이동 평균', linewidth=2)

ax.set_xlabel('에피소드')
ax.set_ylabel('점수')
ax.set_title('Pixelcopter REINFORCE 훈련 곡선')
ax.legend()
plt.tight_layout()
plt.show()


---
## 10. 에이전트 평가

> 📝 **보충 코드**: 원본 노트북에 Pixelcopter 평가 코드가 없어 추가했습니다.  
> 구버전 gym API에 맞게 작성되었습니다.

10번의 에피소드로 평균 보상을 측정합니다.


In [ ]:
def evaluate_agent(env, max_steps, n_eval_episodes, policy):
    """
    n_eval_episodes 동안 에이전트를 실행하여 평균 보상 반환
    (구버전 gym API 사용: reset() → state, step() → state, reward, done, info)

    Args:
        env: 평가 환경 (gym 구버전)
        max_steps: 에피소드당 최대 스텝
        n_eval_episodes: 평가 에피소드 수
        policy: REINFORCE 정책 네트워크
    """
    episode_rewards = []

    for episode in range(n_eval_episodes):
        # ⚠️ 구버전 gym: reset()은 state만 반환
        state = env.reset()
        total_reward = 0.0

        for step in range(max_steps):
            action, _ = policy.act(state)
            # ⚠️ 구버전 gym: step()은 (state, reward, done, info) 반환
            state, reward, done, _ = env.step(action)
            total_reward += reward
            if done:
                break

        episode_rewards.append(total_reward)

    return np.mean(episode_rewards), np.std(episode_rewards)


mean_reward, std_reward = evaluate_agent(
    eval_env,
    pixelcopter_hyperparameters['max_t'],
    pixelcopter_hyperparameters['n_evaluation_episodes'],
    pixelcopter_policy
)
print(f'평균 보상: {mean_reward:.2f} ± {std_reward:.2f}')


---
## 11. 훈련 과정 영상 저장

> 📝 **보충 코드**: 원본 노트북에 Pixelcopter 영상 저장 코드가 없어 추가했습니다.

Pixelcopter는 `render_mode` 파라미터를 지원하지 않으므로,  
`env.render()` 반환값을 직접 사용합니다.


In [ ]:
def record_video_pixelcopter(env_id, policy, video_path, max_steps=10000, fps=30):
    """
    Pixelcopter 환경에서 현재 정책으로 1 에피소드를 실행하여 mp4로 저장
    (구버전 gym API + gym-pygame render 방식 사용)

    Args:
        env_id: 환경 ID
        policy: REINFORCE 정책 네트워크
        video_path: 저장할 mp4 파일 경로
        max_steps: 최대 스텝 수
        fps: 영상 프레임레이트
    """
    record_env = gym.make(env_id)
    frames = []

    # ⚠️ 구버전 gym: reset()은 state만 반환
    state = record_env.reset()

    # gym-pygame은 render()가 픽셀 배열을 반환
    frame = record_env.render(mode='rgb_array')
    if frame is not None:
        frames.append(frame)

    for _ in range(max_steps):
        action, _ = policy.act(state)
        # ⚠️ 구버전 gym: step()은 (state, reward, done, info) 반환
        state, reward, done, _ = record_env.step(action)

        frame = record_env.render(mode='rgb_array')
        if frame is not None:
            frames.append(frame)
        if done:
            break

    record_env.close()

    if len(frames) == 0:
        print('⚠️  프레임이 수집되지 않았습니다. 가상 디스플레이가 실행 중인지 확인하세요.')
        return

    imageio.mimsave(video_path, [np.array(f) for f in frames], fps=fps)
    print(f'✅ 영상 저장 완료: {video_path}  ({len(frames)} 프레임)')


# 훈련 완료 후 영상 저장
video_path = f'{VIDEO_DIR}/pixelcopter_trained.mp4'
record_video_pixelcopter(env_id, pixelcopter_policy, video_path)

# 노트북에서 바로 재생
display(Video(video_path, embed=True, width=400))


---
## 12. Hugging Face Hub 업로드

훈련된 정책 네트워크를 HF Hub에 업로드합니다.

### 사전 준비
1. [Hugging Face 계정 생성](https://huggingface.co/join)
2. [쓰기(write) 권한 토큰 발급](https://huggingface.co/settings/tokens)


In [ ]:
from huggingface_hub import login

# ✏️ 본인의 HF 토큰으로 교체하세요
# ⚠️ 토큰은 절대 외부에 공개하지 마세요!
login(token='hf_xxxxxxxxxxxxxxxxxxxxxxxx')


In [ ]:
def push_to_hub(repo_id, model, hyperparameters, eval_env, video_fps=30):
    """
    평가 → 영상 생성 → HF Hub 업로드 전체 파이프라인
    (구버전 gym API 환경용)

    Args:
        repo_id: HF Hub 저장소 ID
        model: 업로드할 정책 네트워크
        hyperparameters: 훈련 하이퍼파라미터 딕셔너리
        eval_env: 평가 환경
        video_fps: 영상 프레임레이트
    """
    _, repo_name = repo_id.split('/')
    api = HfApi()

    repo_url = api.create_repo(repo_id=repo_id, exist_ok=True)

    with tempfile.TemporaryDirectory() as tmpdirname:
        local_dir = Path(tmpdirname)

        # Step 1: 모델 저장
        torch.save(model, local_dir / 'model.pt')

        # Step 2: 하이퍼파라미터 JSON 저장
        with open(local_dir / 'hyperparameters.json', 'w') as f:
            json.dump(hyperparameters, f)

        # Step 3: 평가 결과 저장
        mean_reward, std_reward = evaluate_agent(
            eval_env, hyperparameters['max_t'],
            hyperparameters['n_evaluation_episodes'], model
        )
        with open(local_dir / 'results.json', 'w') as f:
            json.dump({
                'env_id': hyperparameters['env_id'],
                'mean_reward': mean_reward,
                'n_evaluation_episodes': hyperparameters['n_evaluation_episodes'],
                'eval_datetime': datetime.datetime.now().isoformat(),
            }, f)

        # Step 4: 모델 카드 메타데이터
        env_name = hyperparameters['env_id']
        metadata = {'tags': [env_name, 'reinforce', 'reinforcement-learning',
                             'custom-implementation', 'deep-rl-class']}
        eval_meta = metadata_eval_result(
            model_pretty_name=repo_name,
            task_pretty_name='reinforcement-learning',
            task_id='reinforcement-learning',
            metrics_pretty_name='mean_reward',
            metrics_id='mean_reward',
            metrics_value=f'{mean_reward:.2f} +/- {std_reward:.2f}',
            dataset_pretty_name=env_name,
            dataset_id=env_name,
        )
        metadata = {**metadata, **eval_meta}

        readme_path = local_dir / 'README.md'
        readme_path.write_text(
            f'# **Reinforce** Agent playing **{env_name}**\n'
            f'Unit 4 of the Deep Reinforcement Learning Course\n',
            encoding='utf-8'
        )
        metadata_save(readme_path, metadata)

        # Step 5: 영상 생성
        record_video_pixelcopter(
            env_name, model, local_dir / 'replay.mp4', fps=video_fps
        )

        # Step 6: HF Hub 업로드
        api.upload_folder(
            repo_id=repo_id,
            folder_path=local_dir,
            path_in_repo='.'
        )
        print(f'\n✅ 업로드 완료: {repo_url}')
        print(f'   평균 보상: {mean_reward:.2f} ± {std_reward:.2f}')


In [ ]:
# ✏️ 본인의 HF 사용자명으로 변경하세요.
repo_id = 'YOUR_HF_USERNAME/Reinforce-Pixelcopter-PLE-v0'

push_to_hub(
    repo_id=repo_id,
    model=pixelcopter_policy,
    hyperparameters=pixelcopter_hyperparameters,
    eval_env=eval_env,
    video_fps=30
)
